초급 프로젝트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import pandas as pd
from PIL import Image
import unicodedata  # 0번 섹션에 추가 필요
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# 경로는 환경에 맞게 수정
# train_images, test_images
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/") # 압축을 풀 폴더

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

# merged_annotation json 경로
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

############################################################
# 1. 병합된 JSON 파일을 읽어서 DataFrame으로 만들기
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 1) 이미지 정보 매핑 (id -> file_name)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}

    records = []
    # 2) 어노테이션 순회
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        
        if file_name is None: continue
        
        img_path = os.path.join(img_dir, file_name)
        
        # 실제 이미지 파일이 있는지 확인 (선택 사항이지만 안전함)
        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0], # 파일명을 ID로 사용
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    return pd.DataFrame(records)

# 실행
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
print(f"✅ 학습 데이터 로드 완료: {len(df)} 개의 객체 탐지됨")

✅ 학습 데이터 로드 완료: 4526 개의 객체 탐지됨


In [ ]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

고유 category_id 개수: 73
num_classes (background 포함): 74


In [ ]:
############################################################
# 3. Dataset 정의
############################################################

class OralDrugDataset(Dataset):
    """
    Faster R-CNN용 Dataset
    - __getitem__은 (image, target) 반환
    - image: [C,H,W] float tensor
    - target: dict(boxes, labels, image_id, ...)
    """
    def __init__(self, df, orig2model, transforms=None):
        self.df = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms

        # 이미지 단위로 그룹을 만들기 위해 고유 image_id 리스트를 유지
        self.image_ids = self.df["image_id"].unique().tolist()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # 1) image_id 선택
        image_id = self.image_ids[idx]

        # 2) 해당 image_id에 해당하는 모든 row (여러 객체) 가져오기
        df_img = self.df[self.df["image_id"] == image_id]

        # 3) 이미지 로드
        img_path = df_img["image_path"].iloc[0]
        image = Image.open(img_path).convert("RGB")

        # 4) bbox / labels 구성
        boxes = []
        labels = []

        for _, row in df_img.iterrows():
            x = row["bbox_x"]
            y = row["bbox_y"]
            w = row["bbox_w"]
            h = row["bbox_h"]

            # [x1, y1, x2, y2] 형식으로 변환
            x1 = x
            y1 = y
            x2 = x + w
            y2 = y + h
            boxes.append([x1, y1, x2, y2])

            # 원본 category_id → 모델용 label로 변환
            orig_cat = int(row["category_id"])
            model_cat = self.orig2model[orig_cat]
            labels.append(model_cat)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            # image_id는 loss에는 크게 영향 없음, 그냥 인덱스 정도로 넣어도 무방
            "image_id": torch.tensor([idx]),
        }

        if self.transforms is not None:
            image = self.transforms(image)

        return image, target


############################################################
# 4. Transform, Dataset, DataLoader 구성
############################################################

train_transforms = T.Compose([
    # 필요하다면 여기서 RandomHorizontalFlip 등 augmentation 추가
    T.ToTensor(),  # PIL 이미지를 [0,1] float tensor로 변환
])

dataset = OralDrugDataset(df, orig2model=orig2model, transforms=train_transforms)

# 간단히 train/val split (예: 90% train, 10% val)
indices = torch.randperm(len(dataset)).tolist()
split = int(0.9 * len(indices))
train_indices = indices[:split]
val_indices   = indices[split:]

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset   = torch.utils.data.Subset(dataset, val_indices)

def collate_fn(batch):
    # detection 모델용 collate_fn: 리스트의 튜플을 튜플의 리스트로
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=2,              # GPU 메모리에 맞게 조정
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

print("train steps:", len(train_loader), "val steps:", len(val_loader))



train steps: 670 val steps: 75


In [ ]:

############################################################
# 5. 모델 정의 (Faster R-CNN + ResNet50 FPN 전이학습)
############################################################

# 사전학습된 Faster R-CNN 모델 로드
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

# 분류 head를 우리 데이터셋 클래스 개수에 맞게 교체
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(DEVICE)

# 옵티마이저 설정
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

# (선택) 러닝레이트 스케줄러
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


############################################################
# 6. 학습 루프
############################################################

num_epochs = 5  # 혹은 원하는 에폭 수
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

for epoch in range(num_epochs):
    ##########################
    # 1) Train
    ##########################
    model.train()
    train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        train_loss_sum += losses.item()

    avg_train_loss = train_loss_sum / max(1, len(train_loader))

    ##########################
    # 2) Validation (loss만 측정)
    ##########################
    model.train()  # ★ 여기 중요: eval()이 아니라 train() 상태에서 호출해야 loss 나옴
    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss_sum += losses.item()

    avg_val_loss = val_loss_sum / max(1, len(val_loader))

    print(f"[Epoch {epoch+1}/{num_epochs}] "
          f"train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f}")

    # 스케줄러 한 스텝
    scheduler.step()

# 학습된 모델 저장 (원하면)
torch.save(model.state_dict(), "fasterrcnn_oral_drug.pth")
print("모델 저장 완료")



Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 241MB/s] 


[Epoch 1/5] train_loss: 0.6681 | val_loss: 0.3017
[Epoch 2/5] train_loss: 0.3909 | val_loss: 0.2302
[Epoch 3/5] train_loss: 0.3283 | val_loss: 0.2843
[Epoch 4/5] train_loss: 0.2472 | val_loss: 0.1231
[Epoch 5/5] train_loss: 0.2009 | val_loss: 0.1064
모델 저장 완료


In [ ]:

############################################################
# 7. test_images에 대해 예측 → submission.csv 생성
############################################################

# test 이미지 파일 목록 가져오기
test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")]
test_files = sorted(test_files)

model.eval()

rows = []
annotation_id = 1      # submission용 annotation_id 시작
score_threshold = 0.3  # 너무 낮은 점수는 제거 (필요에 따라 조정)

with torch.no_grad():
    for f in test_files:
        img_path = os.path.join(TEST_IMG_DIR, f)
        image = Image.open(img_path).convert("RGB")

        # image_id = 파일명에서 확장자 제거한 문자열 그대로 사용
        image_id = os.path.splitext(f)[0]

        img_tensor = T.ToTensor()(image).to(DEVICE)
        outputs = model([img_tensor])[0]

        keep = outputs["scores"].cpu() >= score_threshold
        boxes  = outputs["boxes"].cpu()[keep]
        labels = outputs["labels"].cpu()[keep]
        scores = outputs["scores"].cpu()[keep]

        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.tolist()
            w = x2 - x1
            h = y2 - y1

            orig_cat = model2orig[int(lab.item())]

            rows.append({
                "annotation_id": annotation_id,
                "image_id": image_id,  # 문자열 그대로 사용
                "category_id": orig_cat,
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": w,
                "bbox_h": h,
                "score": float(sc.item()),
            })

            annotation_id += 1

# DataFrame으로 만들고 저장
df_sub = pd.DataFrame(rows, columns=[
    "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])
# 이미지 ID별로 점수 높은 순 정렬 후 상위 4개 추출
df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
df_sub = df_sub.groupby("image_id").head(4)

# 3) 최종 annotation_id 부여 (1부터 순차적으로)
df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

# 4) CSV 저장
output_path = "final_submission.csv"
df_sub.to_csv(os.path.join(extract_path, output_path), index=False)

print(f"✅ 생성 완료: {output_path}")
print(f"📊 총 예측 객체 수: {len(df_sub)}")
print(df_sub.head())

✅ 생성 완료: final_submission.csv
📊 총 예측 객체 수: 3257
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1        16550  556.429443   71.300697  399.382629   
1              2        1         1899  159.593399  249.843170  201.259079   
2              3        1        27925  597.204590  675.783875  257.736023   
3              4        1        24849  169.360229  743.308411  183.695557   
4              5       10        16547  104.827522  809.451233  237.359764   

       bbox_h     score  
0  403.272270  0.981536  
1  127.119537  0.980955  
2  475.985046  0.963797  
3  289.701111  0.941548  
4  232.094421  0.986859  


In [ ]:
############################################################
# 8. 모델 성능 평가 (mAP 측정)
############################################################

import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. df_sub(Pandas)를 COCO 평가용 리스트로 변환
eval_results = []
for _, row in df_sub.iterrows():
    eval_results.append({
        "image_id": int(row["image_id"]),
        "category_id": int(row["category_id"]),
        "bbox": [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
        "score": float(row["score"])
    })

# 2. 임시 JSON 파일로 저장
temp_json = "temp_eval.json"
with open(temp_json, "w") as f:
    json.dump(eval_results, f)

# 3. COCO 평가 실행
coco_gt = COCO(TEST_JSON_PATH)
coco_dt = coco_gt.loadRes(temp_json)

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.29s).
Accumulating evaluation results...
DONE (t=0.48s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.374
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.397
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.395
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.374
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.898
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.898
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe